# NoSQL Databases - Projet
Exploration et Interrogation de Bases de Données NoSQL avec Python
---
This is the Neo4J part (see from page 4)

# Imports

In [1]:
pip install neo4j


import os
from dotenv import load_dotenv
import json
from neo4j import GraphDatabase

Note: you may need to restart the kernel to use updated packages.


# CREDS PARSER

In [6]:
# Load credentials from file
creds = {}
with open("creds.txt") as f:
    for line in f:
        key, value = line.strip().split("=", 1)
        creds[key] = value

uri = creds["NEO4J_URI"]
username = creds["NEO4J_USERNAME"]
password = creds["NEO4J_PASSWORD"]
database = creds["NEO4J_DATABASE"]

In [2]:
# In jupyter we can't create a ".env" file, but creds.txt should be .env
# Then we should use this as the parser:

load_dotenv(".env")

uri = os.getenv("NEO4J_URI")
username = os.getenv("NEO4J_USERNAME")
password = os.getenv("NEO4J_PASSWORD")
database = os.getenv("NEO4J_DATABASE")

## Connection test

In [7]:
driver = GraphDatabase.driver(uri, auth=(username, password))

with driver.session(database=database) as session:
    result = session.run("RETURN 'Connexion OK' AS message")
    for record in result:
        print(record["message"])

driver.close()

Connexion OK


# Neo4j
## 0. Import json into neo4j

In [14]:
driver = GraphDatabase.driver(uri, auth=(username, password))

def import_movie(tx, movie):

    tx.run("""
    MERGE (f:Film {id:$id})
    SET f.title=$title,
        f.year=$year,
        f.votes=$votes,
        f.revenue=$revenue,
        f.rating=$rating,
        f.director=$director,
        f.actors=$actors,
        f.genre=$genre
    """,
    id=movie["_id"],
    title=movie.get("title"),
    year=movie.get("year"),
    votes=movie.get("Votes"),
    revenue=movie.get("Revenue (Millions)"),
    rating=movie.get("rating"),
    director=movie.get("Director"),
    actors=movie.get("Actors"),
    genre=movie.get("genre")
    )

with open("movies.json") as file:

    with driver.session() as session:

        count = 0

        for line in file:

            movie = json.loads(line)

            # ignorer les lignes qui ne sont pas des films
            if "title" not in movie:
                count+=1
                continue

            session.execute_write(import_movie, movie)
        print(f"{count} entries were skipped")

driver.close()

2 entries were skipped


## 1. Create `Actor` nodes
`Actors` type nodes containing only distinct actors

In [17]:
driver = GraphDatabase.driver(uri, auth=(username, password))

def create_actors(tx):

    tx.run("""
    MATCH (f:Film)
    WHERE f.actors IS NOT NULL
    WITH f, split(f.actors, ",") AS actors
    UNWIND actors AS actor
    MERGE (:Actor {name: trim(actor)})
    """)

with driver.session() as session:
    session.execute_write(create_actors)

driver.close()

print("Actor nodes created")

Actor nodes created


## 2. Create `ACTED_IN` relationships
Create relationships between actors and movies

In [18]:
driver = GraphDatabase.driver(uri, auth=(username, password))

def create_actor_relationships(tx):

    tx.run("""
    MATCH (f:Film)
    WHERE f.actors IS NOT NULL
    WITH f, split(f.actors, ",") AS actors
    UNWIND actors AS actor
    MATCH (a:Actor {name: trim(actor)})
    MERGE (a)-[:ACTED_IN]->(f)
    """)

with driver.session() as session:
    session.execute_write(create_actor_relationships)

driver.close()

print("ACTED_IN relationships created")

ACTED_IN relationships created


## 3. Create `Director` nodes
Create `Director` nodes from `director`

In [19]:
driver = GraphDatabase.driver(uri, auth=(username, password))

def create_directors(tx):

    tx.run("""
    MATCH (f:Film)
    WHERE f.director IS NOT NULL
    MERGE (:Director {name: f.director})
    """)

with driver.session() as session:
    session.execute_write(create_directors)

driver.close()

print("Director nodes created")

Director nodes created


## 4. Create `DIRECTED` relations

In [20]:
driver = GraphDatabase.driver(uri, auth=(username, password))

def create_director_relationships(tx):

    tx.run("""
    MATCH (f:Film)
    WHERE f.director IS NOT NULL
    MATCH (d:Director {name: f.director})
    MERGE (d)-[:DIRECTED]->(f)
    """)

with driver.session() as session:
    session.execute_write(create_director_relationships)

driver.close()

print("DIRECTED relationships created")

DIRECTED relationships created


## 5. Create `Actor` nodes for project members

In [21]:
## 5. Add project members

members = [
    "Pierre Garat",
    "Théo Leste",
    "Clara Limousin",
    "Arthur Mennessier",
    "Acsel Parsy"
]

driver = GraphDatabase.driver(uri, auth=(username, password))

def add_members(tx):

    for member in members:

        tx.run("""
        MERGE (a:Actor {name:$name})
        WITH a
        MATCH (f:Film {title:"The Departed"})
        MERGE (a)-[:ACTED_IN]->(f)
        """, name=member)

with driver.session() as session:
    session.execute_write(add_members)

driver.close()

print("Project members added")

Project members added


## Verification

In [22]:
driver = GraphDatabase.driver(uri, auth=(username, password))

def verify(tx):

    result = tx.run("""
    MATCH (a:Actor)-[:ACTED_IN]->(f:Film)
    RETURN a.name AS actor, f.title AS film
    LIMIT 10
    """)

    for record in result:
        print(record["actor"], "->", record["film"])

with driver.session() as session:
    session.execute_read(verify)

driver.close()

Leonardo DiCaprio -> The Departed
Matt Damon -> The Departed
Jack Nicholson -> The Departed
Mark Wahlberg -> The Departed
Pierre Garat -> The Departed
Théo Leste -> The Departed
Clara Limousin -> The Departed
Arthur Mennessier -> The Departed
Acsel Parsy -> The Departed
Matthew McConaughey -> Gold


# TODO
**Question 14 -> 30 (p.5)**